# 01 · EDA of the processed data (Member 1)

Exploratory data analysis for issue #1: **class balance, post length, code-mixing examples, top words per class**
(plus rationale statistics for HateXplain).

**Before running:** `python scripts/prepare_data.py` (from the repo root).
If a dataset has not been prepared yet, this notebook falls back to the tiny synthetic sample in `data/sample/`
so every cell still runs.

> Clear the outputs before committing (`Kernel → Restart & Clear Output`).

In [ ]:
# sys / Path: make "import xhate" work from the notebooks/ folder.
import sys
from pathlib import Path
# Counter: word and class counts.
from collections import Counter
# re: to detect Devanagari characters.
import re

# pandas for tables, matplotlib for plots.
import pandas as pd
import matplotlib.pyplot as plt

# Repo root = parent of notebooks/ (works whether the kernel starts in notebooks/ or the root).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Put the repo root on the import path.
sys.path.insert(0, str(REPO_ROOT))

# Our loader functions (Member 1's public API).
from xhate.data import LABEL_NAMES, load_dataframe, read_jsonl

## 1 · Load the data
Each dataset becomes one DataFrame with a `split` column. Missing datasets fall back to the synthetic sample.

In [ ]:
def load_dataset(name):
    """Return all splits of `name` as one DataFrame, or the synthetic sample if not prepared."""
    try:
        # Concatenate train / val / test into one table.
        return pd.concat([load_dataframe(name, s) for s in ("train", "val", "test")], ignore_index=True)
    except FileNotFoundError:
        # Not prepared yet: use the made-up sample and say so.
        print(f"{name}: processed data not found -> using data/sample/{name}_sample.jsonl")
        # Read the sample file into a DataFrame.
        return pd.DataFrame(read_jsonl(REPO_ROOT / "data" / "sample" / f"{name}_sample.jsonl"))

# Load both datasets.
data = {name: load_dataset(name) for name in ("hasoc", "hatexplain")}
# Show the size of each.
{name: len(df) for name, df in data.items()}

## 2 · Class balance
**Q:** Why does this matter? **A:** If HOF and NOT are very unbalanced, accuracy is misleading; we report
macro-F1 and may need class weights (tell Members 2 and 3).

In [ ]:
# Count NOT / HOF per split for each dataset.
for name, df in data.items():
    # Rows = split, columns = class name, values = counts.
    table = pd.crosstab(df["split"], df["label"].map(lambda y: LABEL_NAMES[y]))
    # Make sure both class columns exist (the sample may lack one class in a split).
    table = table.reindex(columns=LABEL_NAMES, fill_value=0)
    # Add the HOF percentage per split.
    table["HOF %"] = (100 * table["HOF"] / table[LABEL_NAMES].sum(axis=1)).round(1)
    # Print the dataset name and the table.
    print(f"== {name} ==")
    display(table)

In [ ]:
# One bar chart per dataset with the overall class counts.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
# Draw each dataset in its own subplot.
for ax, (name, df) in zip(axes, data.items()):
    # Count each class and name it.
    counts = df["label"].map(lambda y: LABEL_NAMES[y]).value_counts().reindex(LABEL_NAMES, fill_value=0)
    # Bar chart.
    counts.plot.bar(ax=ax, rot=0, title=name)
# Avoid overlapping labels.
plt.tight_layout()

## 3 · Post length (in tokens)
**Q:** Why look at length? **A:** It tells Member 3 what `max_length` to use (e.g. the 95th percentile).

In [ ]:
# Add a length column (number of cleaned tokens).
for df in data.values():
    df["n_tokens"] = df["tokens"].map(len)

# Summary statistics of length per dataset and class.
for name, df in data.items():
    # Dataset header.
    print(f"== {name} ==")
    # count / mean / std / min / median / 95th percentile / max per class.
    display(df.groupby(df["label"].map(lambda y: LABEL_NAMES[y]))["n_tokens"]
              .describe(percentiles=[0.5, 0.95]).round(1))

In [ ]:
# Histogram of lengths, HOF vs NOT, per dataset.
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
# One subplot per dataset.
for ax, (name, df) in zip(axes, data.items()):
    # Overlay the two classes.
    for y, label_name in enumerate(LABEL_NAMES):
        # Histogram of lengths for this class.
        df.loc[df["label"] == y, "n_tokens"].plot.hist(ax=ax, bins=30, alpha=0.5, label=label_name)
    # Titles, axis label and legend.
    ax.set_title(name); ax.set_xlabel("tokens"); ax.legend()
# Avoid overlapping labels.
plt.tight_layout()

## 4 · Code-mixing
We tag each token as **Devanagari**, **Roman** or **other** (emoji words, placeholders, punctuation) and look at
how scripts mix inside a post. Note: Romanised Hindi ("tum log") also counts as *Roman*, so the examples below
are the most informative view.

In [ ]:
# A token containing any Devanagari character (Unicode block U+0900-U+097F).
DEVANAGARI = re.compile(r"[ऀ-ॿ]")
# A token made of Latin letters (possibly with an apostrophe).
ROMAN = re.compile(r"^[a-z][a-z']*$")

def script_of(token):
    """Return 'devanagari', 'roman' or 'other' for one token."""
    # Any Devanagari character -> Devanagari.
    if DEVANAGARI.search(token):
        return "devanagari"
    # Placeholders and emoji words are not language.
    if token.startswith("<") or token.startswith("emoji_"):
        return "other"
    # Plain Latin letters -> Roman, anything else -> other.
    return "roman" if ROMAN.match(token) else "other"

def post_script(tokens):
    """Classify a whole post: devanagari only, roman only, mixed, or none."""
    # The set of scripts used in the post (ignoring 'other').
    scripts = {script_of(t) for t in tokens} - {"other"}
    # Both scripts -> mixed.
    if scripts == {"devanagari", "roman"}:
        return "mixed"
    # One script (or none) -> that script.
    return scripts.pop() if scripts else "none"

# Share of posts per script type, per dataset.
for name, df in data.items():
    # Script type of every post.
    df["script"] = df["tokens"].map(post_script)
    # Dataset header.
    print(f"== {name} ==")
    # Percentage of posts per script type.
    display((100 * df["script"].value_counts(normalize=True)).round(1).rename("% of posts"))

In [ ]:
# Show a few code-mixed / Romanised Hindi examples from HASOC (raw vs cleaned).
hasoc = data["hasoc"]
# Prefer posts that mix scripts.
examples = hasoc[hasoc["script"] == "mixed"]
# Fall back to Roman-script posts if there are too few mixed ones.
if len(examples) < 5:
    examples = pd.concat([examples, hasoc[hasoc["script"] == "roman"]])
# Show long texts without cutting them off too early.
pd.set_option("display.max_colwidth", 120)
# Raw text, cleaned text and label for 8 examples.
examples[["raw_text", "text", "label"]].head(8)

## 5 · Top words per class
Stop words, placeholders and punctuation are removed so the content words stand out.
**Q:** Why does this matter for explainability? **A:** These are the words we *expect* LIME / attention to
highlight; if a model highlights something unrelated, that is worth discussing in the error analysis.

In [ ]:
# Small English + Hinglish stop-word list (extend as needed).
STOP = set("""
a an the and or but if of to in on at for with is are was were be been it this that these those i you he she we they
me my your our their his her its not no so do does did have has had will would can just as by from up out about
hai hain ho tha thi ka ki ke ko se me mein main mai ye yeh wo woh vo aur bhi to toh kya kyun nahi na hi par
tum tu aap hum ham log ek is us jo ab kar karo raha rahi rahe
""".split())

def top_words(df, label, k=15):
    """Most common content tokens in posts of class `label`."""
    # Count every content token of every post with this label.
    counts = Counter(t for toks in df.loc[df["label"] == label, "tokens"] for t in toks
                     if t not in STOP and not t.startswith("<") and any(ch.isalpha() for ch in t))
    # Return the k most common as (word, count).
    return counts.most_common(k)

# Side-by-side table: top NOT words vs top HOF words, per dataset.
for name, df in data.items():
    # Dataset header.
    print(f"== {name} ==")
    # Build both columns; pad the shorter one so the table is rectangular.
    cols = {LABEL_NAMES[y]: [f"{w} ({c})" for w, c in top_words(df, y)] for y in (0, 1)}
    # Longest column length.
    n = max(len(v) for v in cols.values())
    # Show the table.
    display(pd.DataFrame({k: v + [""] * (n - len(v)) for k, v in cols.items()}))

## 6 · HateXplain rationales
How many tokens per HOF post are marked as rationale, and which words are marked most often.

In [ ]:
# Only HOF posts that have a rationale.
hx = data["hatexplain"]
hx_rat = hx[hx["has_rationale"]]
# Fraction of tokens marked (hard rationale) per post.
frac = hx_rat["rationale"].map(lambda r: sum(r) / len(r) if r else 0)
# Print summary stats of that fraction.
print("fraction of tokens marked as rationale per HOF post:")
display(frac.describe().round(3))
# Count the words that are marked as rationale.
rat_words = Counter(t for toks, r in zip(hx_rat["tokens"], hx_rat["rationale"]) for t, m in zip(toks, r) if m)
# Show the 20 most common rationale words.
pd.DataFrame(rat_words.most_common(20), columns=["rationale word", "count"])

## 7 · Notes for the team
Write down the key findings here (class imbalance, suggested `max_length`, interesting code-mixing patterns)
and share them in the issue so Members 2-4 can use them.